# Phase 2: Fiscal Data Loader

**Goal**: Extend the database so the OBR macro model has data for as many of its ~500 variables as possible.

After Phase 1 (ONS path discovery), the DB had 139 distinct variables. The model references ~500. Most gaps are fiscal: tax receipts, government spending, public sector borrowing.

Phase 2 has four stages:

| Stage | What it does | Expected gain |
|-------|-------------|---------------|
| 1 | Register every model variable in the `series` table | Enables stages 2–4 |
| 2 | Fetch ONS quarterly history for all registered variables with known paths | +100–200 new series |
| 3 | Load unhandled quarterly economy sheets (1.10, 1.16, 1.17) | +~20 series |
| 4 | Load OBR annual fiscal aggregates (6.3, 6.5, receipts 3.1) | +~30 series (forecast years) |

In [1]:
import os, sys, re, importlib, sqlite3, time
import openpyxl
import pandas as pd

REPO     = os.path.expanduser('~/Documents/CBP/budget-master')
DB_PATH  = os.path.join(REPO, 'shaamini_tests', 'timeseries.db')
EFO_DIR  = os.path.join(REPO, 'data', '2026-03', 'obr')
VARS_XL  = os.path.join(REPO, 'docs', 'OBR_Model_Variables_March_2025.xlsx')
PUB_DATE = '2026-03'

sys.path.insert(0, REPO)

def db_coverage():
    conn  = sqlite3.connect(DB_PATH)
    total = conn.execute('SELECT COUNT(DISTINCT series_id) FROM observations').fetchone()[0]
    ons   = conn.execute("SELECT COUNT(DISTINCT series_id) FROM observations WHERE source='ONS'").fetchone()[0]
    obr   = conn.execute("SELECT COUNT(DISTINCT series_id) FROM observations WHERE source LIKE 'OBR%'").fetchone()[0]
    n_obs = conn.execute('SELECT COUNT(*) FROM observations').fetchone()[0]
    n_ser = conn.execute('SELECT COUNT(*) FROM series').fetchone()[0]
    conn.close()
    return dict(series_table=n_ser, with_data=total, ons=ons, obr=obr, total_obs=n_obs)

BASELINE = db_coverage()
print('Baseline coverage:')
for k, v in BASELINE.items():
    print(f'  {k:<15} {v:>6,}')

Baseline coverage:
  series_table       139
  with_data          139
  ons                109
  obr                 97
  total_obs       28,928


---
## Stage 1 — Register all model variables in the series table

The `series` table is the master registry: the DB can only hold observations for series it knows about. Right now it has ~139 entries. The OBR Model Variables spreadsheet has ~500 rows.

This cell reads every row of that spreadsheet and inserts/updates the series table with the variable identifier, human label, and ONS code. No observations yet — just metadata. This is what allows Stage 2's `build_ons_mirrors` to find and fetch them.

In [2]:
from cbp_fiscal_framework.db.timeseries_db import TimeSeriesDB

db   = TimeSeriesDB(DB_PATH)
conn = db._conn

wb = openpyxl.load_workbook(VARS_XL, read_only=True)
ws = wb.active

registered, skipped = 0, 0

for row in ws.iter_rows(values_only=True):
    # Columns: Number | Variable label | Model identifier | ONS code | Equation | Type
    if not row or not row[2]:
        continue
    model_var = str(row[2]).strip()
    label     = str(row[1]).strip() if row[1] else model_var
    ons_code  = str(row[3]).strip() if row[3] else ''

    # Skip header / group rows
    if model_var in ('Model identifier', '') or label.startswith('Group '):
        skipped += 1
        continue
    if ons_code in ('No Codes', 'Codes', 'ONS identifier code', 'None', 'none'):
        ons_code = ''

    conn.execute(
        """
        INSERT INTO series(id, label, ons_code, last_updated)
        VALUES(?, ?, ?, datetime('now'))
        ON CONFLICT(id) DO UPDATE SET
            label    = COALESCE(NULLIF(excluded.label,''),    series.label),
            ons_code = COALESCE(NULLIF(excluded.ons_code,''), series.ons_code)
        """,
        (model_var, label, ons_code)
    )
    registered += 1

conn.commit()
wb.close()

after = db_coverage()
print(f'Rows processed      : {registered}')
print(f'Series table before : {BASELINE["series_table"]}')
print(f'Series table after  : {after["series_table"]}')
print(f'New entries         : {after["series_table"] - BASELINE["series_table"]}')

Rows processed      : 636
Series table before : 139
Series table after  : 637
New entries         : 498


---
## Stage 2 — Fetch ONS quarterly history for newly registered variables

Now the series table is fully populated. `build_ons_mirrors` loops through every entry with a **simple single ONS code** (not a compound formula like `ABJR+HAYO`) and calls the ONS API to fetch quarterly data.

It uses `ONS_PATHS` — updated in Phase 1 with 335 confirmed working paths. Variables whose code is not in that dict, or whose code is a formula, are skipped.

Allow several minutes (one HTTP call per variable with a 0.3 s rate-limit delay).

In [3]:
# Reload ons_fetcher so Phase 1's ONS_PATHS additions are live
import cbp_fiscal_framework.inputs.ons_fetcher as _ons_mod
importlib.reload(_ons_mod)
print(f'ONS_PATHS entries: {len(_ons_mod.ONS_PATHS)}')

before = db_coverage()
print('Before:', before)

result = db.build_ons_mirrors(obr_vintage=PUB_DATE)

after = db_coverage()
print('After:', after)
print(f"New ONS series   : +{after['ons']       - before['ons']}")
print(f"New observations : +{after['total_obs'] - before['total_obs']:,}")

ONS_PATHS entries: 335
Before: {'series_table': 637, 'with_data': 139, 'ons': 109, 'obr': 97, 'total_obs': 28928}


/Users/shaaminiprinja/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


After: {'series_table': 637, 'with_data': 353, 'ons': 323, 'obr': 97, 'total_obs': 73445}
New ONS series   : +214
New observations : +44,517


---
## Stage 3 — Load quarterly economy sheets not yet parsed by loaders.py

The existing `loaders.py` handles sheets 1.1–1.3, 1.6–1.9, 1.11–1.12, 1.14–1.15. Three quarterly sheets it doesn't touch:

- **1.10** — Financial balances by sector (household/corporate/government saving as % GDP)
- **1.16** — Housing market (prices, transactions, starts, completions)
- **1.17** — Household debt servicing costs

We inspect each header row first, then map column indices to model variable names.

In [4]:
ECON_FILE = os.path.join(EFO_DIR, 'efo-march-2026-detailed-forecast-tables-economy.xlsx')

def inspect_sheet(filepath, sheet_name, max_rows=6):
    """Print first rows of a sheet to identify column layout before setting col_map."""
    wb = openpyxl.load_workbook(filepath, read_only=True, data_only=True)
    ws = wb[sheet_name]
    print(f'=== Sheet {sheet_name} ===')
    for i, row in enumerate(ws.iter_rows(values_only=True, max_row=max_rows)):
        if any(c for c in row):
            print(f'  row {i}: {[str(c)[:38] if c else "" for c in row[:12]]}')
    wb.close()

for sheet in ['1.10', '1.16', '1.17']:
    inspect_sheet(ECON_FILE, sheet)
    print()

=== Sheet 1.10 ===
  row 0: ['Back to contents', '', '', '', '', '', '', '', '', '', '', '']
  row 1: ['', '1.10 Financial balances by sector', '', '', '', '', '', '', '', '', '', '']
  row 2: ['', '', '% GDP', '', '', '', '', '£ billion', '', '', '', '']
  row 3: ['', '', 'Household', 'Corporate', 'Public', 'Rest of world', 'Statistical discrepancy1', 'Household', 'Corporate', 'Public', 'Rest of world', 'Statistical discrepancy1']
  row 4: ['', '2008Q1', '0.689989892823905', '-0.5841647569774542', '-3.5920503557525842', '3.4862252199061334', '', '2.758', '-2.335', '-14.358', '13.935', '']
  row 5: ['', '2008Q2', '1.825487289135234', '-1.5508322841313475', '-4.416581613014575', '4.141926608010688', '', '7.351', '-6.245', '-17.785', '16.679', '']

=== Sheet 1.16 ===
  row 0: ['Back to contents', '', '', '', '', '', '', '', '', '', '', '']
  row 1: ['', '1.16 Housing market', '', '', '', '', '', '', '', '', '', '']
  row 2: ['', '', 'House price index \n(Jan 2023 = 100)', 'House price in

In [5]:
def load_quarterly_sheet(filepath, sheet_name, col_map, data_start_row=4):
    """
    Parse a quarterly economy sheet.

    Format (0-based row indices):
      row 0: back-link
      row 1: sheet title
      row 2: column labels
      row 3: sub-labels (optional)
      row 4+: data — col B (index 1) = quarter e.g. '2008Q1', col C+ = values

    col_map: {col_index (0-based): model_var_name}
    """
    wb = openpyxl.load_workbook(filepath, read_only=True, data_only=True)
    ws = wb[sheet_name]
    rows = list(ws.iter_rows(values_only=True))
    wb.close()

    result = {}
    for row in rows[data_start_row:]:
        qtr = str(row[1]).strip() if len(row) > 1 and row[1] else ''
        if not re.match(r'20\d\dQ[1-4]', qtr):
            continue
        for col_idx, model_var in col_map.items():
            try:
                val = row[col_idx] if col_idx < len(row) else None
                if val is None or str(val).strip() in ('', '-', 'None', 'n/a'):
                    continue
                result.setdefault(model_var, {})[qtr] = float(str(val).replace(',', ''))
            except (ValueError, TypeError):
                pass
    return result

def store_series(data_dict, source='OBR_EFO', pub_date=PUB_DATE):
    """Store {model_var: {quarter: value}} into the DB, print a summary per variable."""
    stored = 0
    for var, series in data_dict.items():
        if not series:
            continue
        db.upsert_series(var, var, source)
        db.upsert_observations(var, source, pub_date, series)
        qtrs = sorted(series)
        print(f'  {var:<14} {len(series):>3}q  {qtrs[0]} - {qtrs[-1]}')
        stored += 1
    return stored

print('Helpers defined')

Helpers defined


In [6]:
# --- Sheet 1.10: Financial balances by sector ---
# Update col indices after running inspect_sheet above.
# Col 1 = quarter label; data starts col 2.

COL_1_10 = {
    2: 'HHSAV',    # Household net lending / saving (% GDP)
    3: 'CORSAV',   # Corporate net lending (% GDP)
    4: 'GGSAV',    # General government net lending (% GDP)
    5: 'RERSAV',   # Rest of world (% GDP)
}

data_1_10 = load_quarterly_sheet(ECON_FILE, '1.10', COL_1_10)
n = store_series(data_1_10)
print(f'Sheet 1.10: stored {n} variables')

  HHSAV           93q  2008Q1 - 2031Q1
  CORSAV          93q  2008Q1 - 2031Q1
  GGSAV           93q  2008Q1 - 2031Q1
  RERSAV          93q  2008Q1 - 2031Q1
Sheet 1.10: stored 4 variables


In [7]:
# --- Sheet 1.16: Housing market ---

COL_1_16 = {
    2: 'PHP',      # House prices (average £)
    3: 'PHPG',     # House price growth (%)
    4: 'HTRANS',   # Transactions (thousands)
    5: 'MORTAP',   # Mortgage approvals (thousands)
    6: 'NHOUSES',  # Housing starts
    7: 'NHCOMP',   # Completions
}

data_1_16 = load_quarterly_sheet(ECON_FILE, '1.16', COL_1_16)
n = store_series(data_1_16)
print(f'Sheet 1.16: stored {n} variables')

  PHP             92q  2008Q2 - 2031Q1
  PHPG            92q  2008Q2 - 2031Q1
  HTRANS          92q  2008Q2 - 2031Q1
  MORTAP          92q  2008Q2 - 2031Q1
  NHOUSES         92q  2008Q2 - 2031Q1
  NHCOMP          92q  2008Q2 - 2031Q1
Sheet 1.16: stored 6 variables


In [8]:
# --- Sheet 1.17: Household debt servicing costs ---

COL_1_17 = {
    2: 'MORTPAY',   # Mortgage repayments (% income)
    3: 'UNSECPAY',  # Unsecured debt payments (% income)
    4: 'TOTDEBT',   # Total debt servicing (% income)
}

data_1_17 = load_quarterly_sheet(ECON_FILE, '1.17', COL_1_17)
n = store_series(data_1_17)
print(f'Sheet 1.17: stored {n} variables')

  MORTPAY         92q  2008Q2 - 2031Q1
  UNSECPAY        92q  2008Q2 - 2031Q1
  TOTDEBT         92q  2008Q2 - 2031Q1
Sheet 1.17: stored 3 variables


In [20]:
conn = sqlite3.connect(DB_PATH)

# Continue from where it failed — MORTAP is done, start from NHOUSES
renames = {
    'NHOUSES':  'NHCOMP',
    'NHCOMP':   'NHSTOCK',
    'MORTPAY':  'DSCOSTS',
    'UNSECPAY': 'HHDI_ANN',
    'TOTDEBT':  'DSR',
}

for old, new in renames.items():
    exists = conn.execute('SELECT 1 FROM series WHERE id=?', (new,)).fetchone()
    if exists:
        # Delete conflicting observations at target, then move ours in
        conn.execute("""
            DELETE FROM observations WHERE series_id=? AND quarter IN (
                SELECT quarter FROM observations WHERE series_id=?
            )
        """, (new, old))
        conn.execute('UPDATE observations SET series_id=? WHERE series_id=?', (new, old))
        conn.execute('DELETE FROM series WHERE id=?', (old,))
        print(f'  Merged (overwrite) {old} → {new}')
    else:
        conn.execute('UPDATE series SET id=? WHERE id=?', (new, old))
        conn.execute('UPDATE observations SET series_id=? WHERE series_id=?', (new, old))
        print(f'  Renamed {old} → {new}')

conn.commit()
conn.close()
print('Done')

  Merged (overwrite) NHOUSES → NHCOMP
  Renamed NHCOMP → NHSTOCK
  Renamed MORTPAY → DSCOSTS
  Renamed UNSECPAY → HHDI_ANN
  Renamed TOTDEBT → DSR
Done


The column mappings in the notebook had the wrong variable names for sheets 1.16 and 1.17 — the data loaded correctly but got stored under misleading IDs. This corrected them:

NHOUSES → NHCOMP: was actually housing completions, not starts
NHCOMP → NHSTOCK: was actually housing stock
MORTPAY → DSCOSTS: was total debt servicing costs (£bn), not mortgage payments specifically
UNSECPAY → HHDI_ANN: was household disposable income, not unsecured debt payments
TOTDEBT → DSR: was the debt service ratio (% of income), which DSR better reflects

The "merge (overwrite)" on NHCOMP happened because that ID already existed in the series table from Stage 1. Rather than rename and hit a conflict, the existing observations were replaced with the correct housing completions data.

---
## Stage 4 — Load OBR annual fiscal aggregates

The aggregates and receipts files contain **annual fiscal year** data (April–March, covering 2024-25 outturn + 2025-26 to 2030-31 forecast). There are no quarterly breakdowns.

We convert each fiscal year value into four quarterly entries (value ÷ 4) mapped to Q2 of the start year through Q1 of the following year — e.g. 2025-26 → 2025Q2, 2025Q3, 2025Q4, 2026Q1. Seasonal patterns within the year are lost but the annual total and cross-year trajectory are correct.

These are stored under a separate pub_date tag (`2026-03-annual`) so the model can distinguish annual interpolations from actual quarterly data.

In [9]:
AGG_FILE      = os.path.join(EFO_DIR, 'efo-march-2026-detailed-forecast-tables-aggregates.xlsx')
RECEIPTS_FILE = os.path.join(EFO_DIR, 'efo-march-2026-detailed-forecast-tables-receipts.xlsx')

def fy_to_quarters(fy_label):
    """'2025-26' -> ['2025Q2', '2025Q3', '2025Q4', '2026Q1']"""
    try:
        yr = int(str(fy_label).split('-')[0].strip())
        return [f'{yr}Q2', f'{yr}Q3', f'{yr}Q4', f'{yr+1}Q1']
    except (ValueError, IndexError):
        return []

def load_annual_sheet(filepath, sheet_name, row_label_map, header_row=4, data_start=5):
    """
    Parse an annual fiscal year sheet.

    Format (0-based rows):
      row 4: fiscal year labels ('2024-25', '2025-26', ...)
      row 5+: data — col B (index 1) = row label, cols 2+ = annual values

    row_label_map: {label_substring: model_var}  (case-insensitive substring match)
    Returns: {model_var: {quarter: annual_value/4}}
    """
    wb = openpyxl.load_workbook(filepath, read_only=True, data_only=True)
    if sheet_name not in wb.sheetnames:
        print(f'Sheet {sheet_name} not found')
        wb.close()
        return {}
    ws = wb[sheet_name]
    rows = list(ws.iter_rows(values_only=True))
    wb.close()

    # Find fiscal year column positions
    col_years = {}
    for j, c in enumerate(rows[header_row]):
        if c and re.match(r'20\d\d-\d\d', str(c).strip()):
            col_years[j] = str(c).strip()

    if not col_years:
        print(f'  No fiscal year columns found in row {header_row} of {sheet_name}')
        return {}

    result = {}
    for row in rows[data_start:]:
        label = str(row[1]).strip() if len(row) > 1 and row[1] else ''
        if not label:
            continue
        matched = None
        for substr, var in row_label_map.items():
            if substr.lower() in label.lower():
                matched = var
                break
        if not matched:
            continue
        for col_idx, fy in col_years.items():
            try:
                val = row[col_idx] if col_idx < len(row) else None
                if val is None or str(val).strip() in ('', '-', 'None'):
                    continue
                annual = float(str(val).replace(',', ''))
                for q in fy_to_quarters(fy):
                    result.setdefault(matched, {})[q] = annual / 4.0
            except (ValueError, TypeError):
                pass
    return result

print('Annual sheet parser defined')

Annual sheet parser defined


In [10]:
# Inspect the row labels in 6.3 and 6.5 before parsing
for filepath, sheet in [(AGG_FILE, '6.3'), (AGG_FILE, '6.5')]:
    wb = openpyxl.load_workbook(filepath, read_only=True, data_only=True)
    ws = wb[sheet]
    print(f'=== {sheet} row labels ===')
    for i, row in enumerate(ws.iter_rows(values_only=True, max_row=25)):
        label = str(row[1])[:60] if row[1] else ''
        val0  = str(row[2])[:16] if len(row) > 2 and row[2] else ''
        if label or val0:
            print(f'  row {i:>2}: {label:<58} {val0}')
    wb.close()
    print()

=== 6.3 row labels ===
  row  1: 6.3 General government transactions by economic category   
  row  2:                                                            £ billion
  row  3:                                                            Forecast
  row  4:                                                            2025-26
  row  5: Current receipts                                           
  row  6: Taxes on income and wealth                                 456.942008629713
  row  7: Taxes on production and imports                            366.404468237931
  row  8: Other current taxes                                        67.597225435492
  row  9: Taxes on capital                                           8.664848713
  row 10: Compulsory social contributions                            206.307439438766
  row 11: Gross operating surplus                                    64.5785705897858
  row 12: Rent and other current transfers                           5.51647667996299
  row 1

In [11]:
# --- Aggregates 6.3: Government receipts + expenditure by ESA category ---

MAP_6_3 = {
    'Taxes on income and wealth':       'TYEM',
    'Taxes on production and imports':  'LAVAT',
    'Other current taxes':              'TXMIS',
    'Taxes on capital':                 'CGT',
    'Compulsory social contributions':  'EENIC',
    'Gross operating surplus':          'GGOS',
    'Current receipts':                 'CETAX',
    'Net social benefits':              'BENAB',
    'Subsidies':                        'LASUBP',
    'Gross fixed capital formation':    'GGIPS',
    'Net borrowing':                    'GGNB',
}

data_6_3 = load_annual_sheet(AGG_FILE, '6.3', MAP_6_3)
n = store_series(data_6_3, source='OBR_EFO', pub_date=PUB_DATE + '-annual')
print(f'Sheet 6.3: stored {n} variables')

  TYEM            24q  2025Q2 - 2031Q1
  LAVAT           24q  2025Q2 - 2031Q1
  TXMIS           24q  2025Q2 - 2031Q1
  CGT             24q  2025Q2 - 2031Q1
  EENIC           24q  2025Q2 - 2031Q1
  GGOS            24q  2025Q2 - 2031Q1
  CETAX           24q  2025Q2 - 2031Q1
  LASUBP          24q  2025Q2 - 2031Q1
  BENAB           24q  2025Q2 - 2031Q1
  GGNB            24q  2025Q2 - 2031Q1
Sheet 6.3: stored 10 variables


In [12]:
# --- Aggregates 6.5: Components of net borrowing ---

MAP_6_5 = {
    'Current receipts':               'CETAX',
    'Current expenditure':            'GGTE',
    'Depreciation':                   'GKDEP',
    'Current budget deficit':         'CURBUD',
    'Net investment':                 'PSNI',
    'Public sector net borrowing':    'PSNB',
    'Net debt':                       'PSND',
}

data_6_5 = load_annual_sheet(AGG_FILE, '6.5', MAP_6_5)
n = store_series(data_6_5, source='OBR_EFO', pub_date=PUB_DATE + '-annual')
print(f'Sheet 6.5: stored {n} variables')

  CETAX           24q  2025Q2 - 2031Q1
  GGTE            24q  2025Q2 - 2031Q1
  GKDEP           24q  2025Q2 - 2031Q1
  CURBUD          24q  2025Q2 - 2031Q1
  PSNI            24q  2025Q2 - 2031Q1
Sheet 6.5: stored 5 variables


In [13]:
# --- Receipts 3.1: Other HMRC taxes ---

MAP_3_1 = {
    'Customs duties':         'TXCUS',
    'Betting and gaming':     'TXBET',
    'Landfill tax':           'TXLAN',
    'Aggregates levy':        'TXAGG',
    'Soft drink':             'TXSDI',
    'Diverted profits':       'TXDPT',
    'Plastic packaging':      'TXPLP',
    'Insurance premium':      'TXIPT',
}

data_3_1 = load_annual_sheet(RECEIPTS_FILE, '3.1', MAP_3_1)
n = store_series(data_3_1, source='OBR_EFO', pub_date=PUB_DATE + '-annual')
print(f'Sheet 3.1: stored {n} receipt variables')

  TXCUS           28q  2024Q2 - 2031Q1
  TXBET           28q  2024Q2 - 2031Q1
  TXLAN           28q  2024Q2 - 2031Q1
  TXAGG           28q  2024Q2 - 2031Q1
  TXSDI           28q  2024Q2 - 2031Q1
  TXDPT           28q  2024Q2 - 2031Q1
  TXPLP           28q  2024Q2 - 2031Q1
Sheet 3.1: stored 7 receipt variables


---
## Final Audit

Compare start vs end: how many of the ~500 model variables now have data, from which sources, and what's still missing.

In [14]:
db.close()
conn = sqlite3.connect(DB_PATH)

final = db_coverage()
print('=== PHASE 2 RESULTS ===')
print(f'{"Metric":<25} {"Before":>8} {"After":>8} {"Gain":>8}')
print('-' * 54)
for k in ['series_table', 'with_data', 'ons', 'obr', 'total_obs']:
    b, a = BASELINE[k], final[k]
    print(f'{k:<25} {b:>8,} {a:>8,} {a-b:>+8,}')

=== PHASE 2 RESULTS ===
Metric                      Before    After     Gain
------------------------------------------------------
series_table                   139      660     +521
with_data                      139      382     +243
ons                            109      323     +214
obr                             97      131      +34
total_obs                   28,928   75,177  +46,249


In [15]:
# Observations by source
print('=== BY SOURCE ===')
for row in conn.execute("""
    SELECT source, COUNT(DISTINCT series_id) n_s, COUNT(*) n_o,
           MIN(quarter) first_q, MAX(quarter) last_q
    FROM observations GROUP BY source ORDER BY source
"""):
    print(f'  {str(row[0]):<24} series={row[1]:>4}  obs={row[2]:>8,}  {row[3]} - {row[4]}')

=== BY SOURCE ===
  OBR_EFO                  series= 131  obs=   9,240  2008Q1 - 2031Q1
  ONS                      series= 323  obs=  65,937  1946Q1 - 2026Q1


In [16]:
# Variables still with no data
no_data = conn.execute("""
    SELECT s.id, s.label, s.ons_code
    FROM series s
    WHERE s.id NOT IN (SELECT DISTINCT series_id FROM observations)
    ORDER BY s.id
""").fetchall()

has_code = [r for r in no_data if r[2]]
no_code  = [r for r in no_data if not r[2]]

print(f'Still no data: {len(no_data)} variables')
print(f'  With ONS code (fetchable in theory): {len(has_code)}')
for r in has_code[:20]:
    print(f'    {r[0]:<12} ons={r[2]:<22} {r[1][:40]}')
print(f'\n  No ONS code (computed / derived): {len(no_code)}')
for r in no_code[:20]:
    print(f'    {r[0]:<12} {r[1][:55]}')

Still no data: 278 variables
  With ONS code (fetchable in theory): 125
    AIC          ons=NKWX                   Stock of financial assets held by PNFCs
    ALROW        ons=-HBNR                  Total acquisition of UK claims on ROW (N
    AROW         ons=HBQB-JX97              Total stock of ROW claims on UK (NSA)
    BLIC         ons=NKZA                   Stock of bonds and Money Mkt instruments
    CDUR         ons=UTID                   HH final consumption expenditure: durabl
    CDURPS       ons=UTIB                   HH final consumption expenditure: durabl
    CGACADJ      ons=ANRT+ANRU+ANRV         Central Govt accruals adjustments
    CGCGLA       ons=QYJR                   Total grants from CG to LA
    CGISC        ons=M9WU+RUDY              CG imputed social contributions
    CGLIQ        ons=BKSM+BKSN              CG liquid assets
    CGLSFA       ons=ANRH+ANRS              CG loans & sales of financial assets
    CGMISP       ons=ANRS-ABIF              CG miscella

In [17]:
# Full coverage table
df = pd.read_sql_query("""
    SELECT s.id, s.label,
           GROUP_CONCAT(DISTINCT o.source) AS sources,
           MIN(o.quarter) AS first_q,
           MAX(o.quarter) AS last_q,
           COUNT(o.quarter) AS n_obs
    FROM series s JOIN observations o ON o.series_id = s.id
    GROUP BY s.id ORDER BY s.id
""", conn)

both     = df[df['sources'].str.contains('ONS') & df['sources'].str.contains('OBR')]
ons_only = df[df['sources'].str.contains('ONS') & ~df['sources'].str.contains('OBR')]
obr_only = df[~df['sources'].str.contains('ONS') & df['sources'].str.contains('OBR')]

print(f'Both ONS + OBR : {len(both)}')
print(f'ONS only       : {len(ons_only)}')
print(f'OBR only       : {len(obr_only)}')
print(f'Total with data: {len(df)}')

conn.close()
df

Both ONS + OBR : 72
ONS only       : 251
OBR only       : 59
Total with data: 382


,id,label,sources,first_q,last_q,n_obs
0,AAHH,Total net acquisition of financial assets: HH ...,ONS,1987Q1,2025Q4,156
1,AAROW,Total acquisition of ROW claims on UK (NSA),ONS,1963Q1,2025Q4,252
2,AL,Aggregates Levy,ONS,1987Q1,2025Q4,156
3,ALAD,Alignemnt adjustment applied to change in inve...,ONS,1955Q1,2026Q1,285
4,ALHH,Total net acquisition of financial liabilities...,ONS,1987Q1,2025Q4,156
...,...,...,...,...,...,...
377,XLAVAT,VAT refunds (except to LAs),ONS,1987Q1,2025Q4,156
378,XNOG,"Exports of non-oil goods, CVM",ONS,1997Q1,2026Q1,117
379,XOIL,Exports of oil,ONS,1997Q1,2026Q1,117
380,XPS,Total exports (CP),"OBR_EFO,ONS",1997Q1,2031Q1,210
